## **Problem Set 3**. Text I/II: classical representations, classification & embeddings
### *Goal: move from sparse count vectors to dense distributional semantics.*

In [7]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import classification_report
import spacy
from sklearn.pipeline import Pipeline

##### 1. **[Theory]** Derive the TF-IDF weight $w_{t,d} = \text{tf}_{t,d} \cdot \log \frac{N}{\text{df}_t}$. Give an information-theoretic reading of the $\log(N/\text{df}_t)$ term and explain why raw term frequency alone is a poor weighting.

TF-IDF (Term Frequency-Inverse Document Frequency)

1. **Term Frequency ($\text{tf}$)**

Term frequency measures how often a term appears within a document:

$$\text{tf}(t, d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}$$

Where:
* $f_{t,d}$ - the number of times term $t$ occurs in document $d$.
* The denominator is the total number of terms in document $d$.

2. **Why raw term frequency alone is a poor weighting?**

- Longer documents naturally repeat words more often, because they have more text, not because the topic is more important. For example, common helper words like **"the"**, **"a"**, or **"and"** will appear hundreds of times in a long text simply due to its size, completely unrelated to what the document is actually about.
- Finding a keyword 50 times doesn't make a document 50 times more useful than finding it just once. After the first few mentions, extra repeats don't add any real value.

3. **Inverse Document Frequency ($\text{idf}$)**

To give higher weight to terms that appear in fewer documents and lower weight to frequent ones, we use Inverse Document Frequency. 

The weight of the term is inversely proportional to the number of documents in which it is presented:
$$w_t \propto \frac{1}{\text{df}_t}$$

Consider a corpus of $N$ documents, where term $t$ appears in $\text{df}_t$ documents (Document Frequency). If we pick a random document from the corpus, the probability $P_t$ that it contains term $t$ is:

$$P_t = \frac{\text{df}_t}{N}$$

Using Shannon's information theory, the self-information of observing term $t$ is:

$$I(t) = -\log P_t = -\log \left( \frac{\text{df}_t}{N} \right) = \log \left( \frac{N}{\text{df}_t} \right) = idf(t)$$

* *High $\text{df}_t$ (popular/common words like "the", "and", "a"):* $P_t \approx 1$, which means $\log(N/\text{df}_t) \to 0$. These terms contribute very little information.
* *Low $\text{df}_t$ (Rare, specific words):* $P_t \ll 1$, meaning $\log(N/\text{df}_t)$ is high. These terms help in discriminating, distinguishing, and ranking documents, contributing more information to the corpus.

4. **TF-IDF Weight**
Combining all together:

$$w_{t,d} = \text{tf}_{t,d} \cdot \log \left( \frac{N}{\text{df}_t} \right)$$

##### 2. **[Theory]** *(Multinomial Naïve Bayes.)* Derive the MAP parameter estimates with Laplace smoothing, $\hat{\theta}_{t|c} = \frac{n_{t,c} + \alpha}{\sum_{t'} n_{t',c} + \alpha |V|}$. Show that the resulting decision rule is *linear* in the word-count vector, and compare its form to logistic regression.

##### 3. **[Theory]** *(Skip-gram with negative sampling.)* Write the SGNS objective $\log \sigma(v_c^\top v_w) + \sum_{i=1}^k \mathbb{E}_{w_i \sim P_n} \log \sigma(-v_{w_i}^\top v_w)$. Derive the gradient w.r.t. a centre-word vector $v_w$, and explain informally why SGNS approximates a shifted PMI factorisation.

##### 4. **[Python]** Build a TF-IDF pipeline ($\text{spaCy/NLTK preprocessing} \to \text{TfidfVectorizer}$) and train both multinomial Naïve Bayes and logistic regression on a text-classification dataset (e.g. 20 Newsgroups subset). Report precision/recall/F1 and list the most informative features per class.

- [The 20 Newsgroups data set](http://qwone.com/~jason/20Newsgroups/)

The 20 Newsgroups data set is a collection of approximately $20 000$ newsgroup documents, partitioned (nearly) evenly across 20 different newsgroups. Some of the newsgroups are very closely related to each other (e.g. `comp.sys.ibm.pc.hardware / comp.sys.mac.hardware`), while others are highly unrelated (e.g `misc.forsale / soc.religion.christian`).

Here is a list of the 20 newsgroups, partitioned (more or less) according to subject matter:

$$
\begin{array}{|l|l|l|}
\hline
\begin{array}{l} 
\text{comp.graphics} \\ 
\text{comp.os.ms-windows.misc} \\ 
\text{comp.sys.ibm.pc.hardware} \\ 
\text{comp.sys.mac.hardware} \\ 
\text{comp.windows.x} 
\end{array} & 
\begin{array}{l} 
\text{rec.autos} \\ 
\text{rec.motorcycles} \\ 
\text{rec.sport.baseball} \\ 
\text{rec.sport.hockey} 
\end{array} & 
\begin{array}{l} 
\text{sci.crypt} \\ 
\text{sci.electronics} \\ 
\text{sci.med} \\ 
\text{sci.space} 
\end{array} \\
\hline
\begin{array}{l} 
\text{misc.forsale} 
\end{array} & 
\begin{array}{l} 
\text{talk.politics.misc} \\ 
\text{talk.politics.guns} \\ 
\text{talk.politics.mideast} 
\end{array} & 
\begin{array}{l} 
\text{talk.religion.misc} \\ 
\text{alt.atheism} \\ 
\text{soc.religion.christian} 
\end{array} \\
\hline
\end{array}
$$


In [5]:
cats = ['alt.atheism', 'sci.space', 'comp.windows.x', 'rec.sport.baseball']
newsgroups_train = fetch_20newsgroups(subset='train', categories=cats)
newsgroups_test = fetch_20newsgroups(subset='test', categories=cats)

# python -m spacy download en_core_web_sm
def spacy_preprocessor(texts):
    nlp = spacy.load("en_core_web_sm")
    preprocessed_texts = []
    for doc in nlp.pipe(texts, disable=["parser", "ner"]):
        tokens = [token.lemma_.lower() for token in doc if not token.is_stop and not token.is_punct]
        preprocessed_texts.append(" ".join(tokens))
    return preprocessed_texts

In [ ]:
def tfidf_pipeline(X_train, y_train, X_test, y_test, target_names):
  X_train_clean = spacy_preprocessor(X_train)
  X_test_clean = spacy_preprocessor(X_test)

  models = {
    'Multinomial Naive Bayes': Pipeline(
          [('tfidf', TfidfVectorizer(max_df=0.9, min_df=2)), ('clf', MultinomialNB())]
      ),
    'Logistic Regression': Pipeline(
          [('tfidf', TfidfVectorizer(max_df=0.9, min_df=2)), ('clf', LogisticRegression(max_iter=1000, C=1.0))]
      ),
  }

  for name, pipeline in models.items():
    pipeline.fit(X_train_clean, y_train)
    y_pred = pipeline.predict(X_test_clean)

    print("\n")
    print(f" == Classification Report for {name} ==")
    print("\n")
    print(classification_report(y_test, y_pred, target_names=target_names))

    vectorizer = pipeline.named_steps['tfidf']
    classifier = pipeline.named_steps['clf']
    feature_names = np.array(vectorizer.get_feature_names_out())

    if hasattr(classifier, 'coef_'):
      weights = classifier.coef_  # Logistic Regression
    elif hasattr(classifier, 'feature_log_prob_'):
      weights = classifier.feature_log_prob_  # Naive Bayes

    print(f' == Top 5 Informative Features for {name} ==')
    for i, class_name in enumerate(target_names):
      top_indices = weights[i].argsort()[-5:][::-1]
      top_features = feature_names[top_indices]
      print(f'Class [{class_name}]: {", ".join(top_features)}')

tfidf_pipeline(
    newsgroups_train.data,
    newsgroups_train.target,
    newsgroups_test.data,
    newsgroups_test.target,
    newsgroups_train.target_names,
)



 == Classification Report for Multinomial Naive Bayes ==


                    precision    recall  f1-score   support

       alt.atheism       0.99      0.95      0.97       319
    comp.windows.x       0.98      0.97      0.97       395
rec.sport.baseball       0.98      0.99      0.98       397
         sci.space       0.95      0.97      0.96       394

          accuracy                           0.97      1505
         macro avg       0.97      0.97      0.97      1505
      weighted avg       0.97      0.97      0.97      1505

 == Top 5 Informative Features for Multinomial Naive Bayes ==
Class [alt.atheism]: edu, god, keith, com, write
Class [comp.windows.x]: window, com, edu, motif, server
Class [rec.sport.baseball]: edu, game, team, year, baseball
Class [sci.space]: space, edu, nasa, henry, access


 == Classification Report for Logistic Regression ==


                    precision    recall  f1-score   support

       alt.atheism       0.99      0.92      0.95       319


##### 5. **[Python]** Train word2vec with `gensim` on the same corpus and separately load pretrained GloVe. For both, return the nearest neighbours of five query words and evaluate on an analogy set (e.g. $\text{king} - \text{man} + \text{woman}$)

##### 6. **[Analysis]** Probe the embeddings for social bias (e.g. gendered analogies or association tests). Report what you find in one paragraph and comment on the risk of propagating such bias downstream.